In [ ]:
import os, sys
sys.path.append('../')
import MeshFEM, mesh, benchmark
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [ ]:
base_path = 'local_exps_with_uv/'

In [ ]:
plot_option = 'uv'
plot_option = 'obj'
plot_option = 'all'

In [ ]:
model_name_list = [entry.name for entry in os.scandir(base_path) if entry.is_dir()]
hessian_option_list = ['Adaptive', 'Always', 'Never']
thread_num_list = [0]

# delete Figs in model_name_list 
if 'Figs' in model_name_list:  model_name_list.remove('Figs')
if 'Videos' in model_name_list:  model_name_list.remove('Videos')

In [ ]:
model_name_list

In [ ]:
user_model_name = 'Octo_cut2'
if user_model_name not in model_name_list:
    raise RuntimeError(f"[Error] '{user_model_name}' is not in directory {base_path} !")

In [ ]:
def getFastestRepeatIndex(directory):
    file_path = os.path.join(directory, "summary.txt")
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"'summary.txt' not found in {directory}")
    # Read the last line of the file
    with open(file_path, "r") as file:
        lines = file.readlines()
        if not lines:  raise RuntimeError(f"'summary.txt' is empty in {directory}")
        return (lines[-1].strip())[-1]

In [ ]:
def read_benchmark_data(directory):
    # Full paths to the files
    pkl_file_path = os.path.join(directory, "benchmark_dict.pkl")
    npz_file_path = os.path.join(directory, "obj_time_gradnorm.npz")
    
    # Check if files exist
    if not os.path.isfile(pkl_file_path):
        raise FileNotFoundError(f"'benchmark_dict.pkl' not found in {directory}")
    if not os.path.isfile(npz_file_path):
        raise FileNotFoundError(f"'obj_and_time.npz' not found in {directory}")

    # Load the dictionary from pickle file
    with open(pkl_file_path, "rb") as f:
        benchmark_dict = pickle.load(f)

    # Load numpy arrays from .npz file
    npz_data = np.load(npz_file_path)
    obj_arr = npz_data['obj_arr']
    time_arr = npz_data['time_arr']
    grad_norm_arr = npz_data['grad_norm_arr']

    return obj_arr, time_arr, grad_norm_arr, benchmark_dict

In [ ]:
def read_uv_min_data(directory):
    largest_i = sum(1 for f in os.listdir(directory) if f.endswith(".npz")) - 1
    uv_fn = 'uv_ravel_iter_' + str(largest_i) + '.npz'
    uv_data = np.load(os.path.join(directory, uv_fn))
    uv_min = uv_data['arr']
    return uv_min

In [ ]:
def compute_uv_distance(directory):
    numUVs = sum(1 for f in os.listdir(directory) if f.endswith(".npz"))
    uv_min = read_uv_min_data(directory)
    
    dist_list = []
    for i in range(numUVs-1):
        uv_fn = 'uv_ravel_iter_' + str(i) + '.npz'
        uv_npz = np.load(os.path.join(directory, uv_fn))
        uv_data = uv_npz['arr']
        dist = np.linalg.norm(uv_data - uv_min)
        dist /= np.linalg.norm(np.max(uv_min, axis=0) - np.min(uv_min, axis=0))
        dist_list.append(dist)
    
    return dist_list

In [ ]:
# Plot different hessian projection options under one thread configuration
def save_obj_grad_time_figure(obj_grad_time_list, thread_num, file_name_wo_ext, directory):
    tn = thread_num
    iterations_list = []  # iteration numbers for each hessian projection option
    for i in range(3):
        iterations = np.arange(0, obj_grad_time_list[i][tn].shape[1])
        iterations_list.append(iterations)
    color_list = ['dodgerblue', 'magenta', 'tomato']
    line_style_list = ['-', '--', '-.']
    
    # Generate plt
    plt.figure(figsize=(12, 12))
    plt.subplot(2,2,1)
    for i in range(3):
        plt.plot(iterations_list[i], obj_grad_time_list[i][tn][0], ls=line_style_list[i], color=color_list[i], label=hessian_option_list[i])
    plt.title(f"Model: {user_model_name}", fontsize=16)
    plt.yscale('log')
    plt.xlabel("Iteration", fontsize=12)
    plt.ylabel(" Energy ", fontsize=14)
    plt.legend()
    
    plt.subplot(2,2,2)
    for i in range(3):
        plt.plot(iterations_list[i], obj_grad_time_list[i][tn][1], ls=line_style_list[i], color=color_list[i], label=hessian_option_list[i])
    plt.title(f"Model: {user_model_name}", fontsize=16)
    plt.yscale('log')
    plt.xlabel("Iteration", fontsize=12)
    plt.ylabel(" Grad Norm ", fontsize=14)
    plt.legend()

    plt.subplot(2,2,3)
    for i in range(3):
        plt.plot(obj_grad_time_list[i][tn][2], obj_grad_time_list[i][tn][0], ls=line_style_list[i], color=color_list[i], label=hessian_option_list[i])

    plt.yscale('log')
    plt.xlabel("Time [sec]", fontsize=12)
    plt.ylabel(" Energy ", fontsize=14)
    plt.legend()

    plt.subplot(2,2,4)
    for i in range(3):
        plt.plot(obj_grad_time_list[i][tn][2], obj_grad_time_list[i][tn][1], ls=line_style_list[i], color=color_list[i], label=hessian_option_list[i])
    plt.yscale('log')
    plt.xlabel("Time [sec]", fontsize=12)
    plt.ylabel("Grad Norm", fontsize=14)
    plt.legend()
    plt.tight_layout()
    
    full_fn = file_name_wo_ext + '_thread' + str(thread_num) + '.png'
    plt.savefig(os.path.join(directory, full_fn), dpi=300)
    print(f"[Plot] '{full_fn}' saved in {directory}!")
    plt.close()

In [ ]:
def save_uv_dist_figure(uv_dist_list, plot_name, directory):
    # Plot different hessian projection options under one thread configuration
    iterations_list = []
    for i in range(3):
        iterations = np.arange(0, len(uv_dist_list[i]))
        iterations_list.append(iterations)

    color_list = ['dodgerblue', 'magenta', 'tomato']
    line_style_list = ['-', '--', '-.']
    
    plt.figure(figsize=(8, 8))
    for i in range(3):
        plt.plot(iterations_list[i], uv_dist_list[i], ls=line_style_list[i], color=color_list[i], label=hessian_option_list[i])
    plt.title(f"Model: {user_model_name}", fontsize=16)
    plt.yscale('log')
    plt.xlabel("Iteration", fontsize=12)
    plt.ylabel(" Distance to Minimum ", fontsize=14)
    plt.legend()
    plt.tight_layout()
    
    full_fn = plot_name + '.png'
    plt.savefig(os.path.join(directory, full_fn), dpi=300)
    print(f"[Plot] '{full_fn}' saved in {directory}!")
    plt.close()

In [ ]:
plot_folder = 'Figs'
plot_dir = os.path.join(base_path, plot_folder)
if not os.path.exists(plot_dir):  os.makedirs(plot_dir)

In [ ]:
obj_grad_time_list = [[], [], []]
uv_dist_list = []

for hessopt_ind, hessian_option in enumerate(hessian_option_list):
    if plot_option == 'all' or plot_option == 'obj':
        # read timing data
        for thread_num in thread_num_list:
            thread_dir_name = 'thread' + '_' + str(thread_num)
            cur_dir = os.path.join(base_path, user_model_name, hessian_option, thread_dir_name)
            # read file 'summary.txt'
            fast_ind = getFastestRepeatIndex(cur_dir)
            if fast_ind == '0':  fast_ind = '10'
            repeat_dir_name = 'repeat' + '_' + fast_ind
            data_dir = os.path.join(cur_dir, repeat_dir_name)
            obj_arr, time_arr, grad_norm_arr, benchmark_dict = read_benchmark_data(data_dir)
            obj_grad_time = np.vstack((obj_arr, grad_norm_arr, time_arr)) # make a (3,n) numpy array
            obj_grad_time_list[hessopt_ind].append(obj_grad_time)
    
    if plot_option == 'all' or plot_option == 'uv':
        # read UV data
        uv_dir_name = 'UVs'
        cur_dir = os.path.join(base_path, user_model_name, hessian_option, uv_dir_name)
        dist_list = compute_uv_distance(cur_dir)
        uv_dist_list.append(dist_list)

if plot_option == 'all' or plot_option == 'obj':
    plot_obj_name = user_model_name + '_objgradvsT'
    for thread_num in thread_num_list:
        save_obj_grad_time_figure(obj_grad_time_list, thread_num, plot_obj_name, plot_dir)

if plot_option == 'all' or plot_option == 'uv':
    plot_uv_name = user_model_name + '_uvdist'
    save_uv_dist_figure(uv_dist_list, plot_uv_name, plot_dir)

In [ ]:
brek

In [ ]:
# Plot different hessian projection options under one thread configuration
iterations_list = []
for i in range(3):
    iterations = np.arange(0, len(uv_dist_list[i]))
    iterations_list.append(iterations)

color_list = ['dodgerblue', 'magenta', 'tomato']
line_style_list = ['-', '--', '-.']

In [ ]:
plt.figure(figsize=(8, 8))
for i in range(3):
    plt.plot(iterations_list[i], uv_dist_list[i], ls=line_style_list[i], color=color_list[i], label=hessian_option_list[i])
plt.title(f"Model: {user_model_name}", fontsize=16)
plt.yscale('log')
plt.xlabel("Iteration", fontsize=12)
plt.ylabel(" Distance to Minimum ", fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()
